# Notebook 06: Final Outputs – Demo-Ready Maps and Summary
**Project:** Parking-Induced Congestion AI System

This notebook generates:
- Final summary table
- Folium heatmap
- Hotspot cluster map
- Enforcement priority map
- Full pipeline explanation

## 1. Import Libraries

In [2]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import folium
from folium.plugins import HeatMap
import os
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid')
print('Libraries imported.')

Libraries imported.


## 2. Define Paths

In [3]:
PARKING_CSV      = '../data/processed/parking_violations.csv'
CLUSTERED_CSV    = '../data/processed/parking_violations_with_clusters.csv'
HOTSPOT_CSV      = '../data/processed/hotspot_clusters.csv'
RISK_CSV         = '../data/processed/congestion_risk_scores.csv'
PRIORITY_CSV     = '../data/processed/enforcement_priority_zones.csv'
PREDICTIONS_CSV  = '../data/processed/ml_predictions.csv'
FINAL_CSV        = '../data/processed/final_summary_table.csv'
MAPS_DIR         = '../data/processed/maps/'
CHARTS_DIR       = '../data/processed/charts/'

os.makedirs(MAPS_DIR, exist_ok=True)
os.makedirs(CHARTS_DIR, exist_ok=True)
print('Paths ready.')

Paths ready.


## 3. Load All Data

In [4]:
def safe_load(path, label):
    try:
        df = pd.read_csv(path, low_memory=False)
        print(f'Loaded {label}: {df.shape}')
        return df
    except Exception as e:
        print(f'WARNING: Could not load {label} from {path}: {e}')
        return pd.DataFrame()

df_parking   = safe_load(PARKING_CSV,   'parking_violations')
df_clustered = safe_load(CLUSTERED_CSV, 'parking_violations_with_clusters')
df_hotspots  = safe_load(HOTSPOT_CSV,   'hotspot_clusters')
df_risk      = safe_load(RISK_CSV,      'congestion_risk_scores')
df_priority  = safe_load(PRIORITY_CSV,  'enforcement_priority_zones')
df_preds     = safe_load(PREDICTIONS_CSV, 'ml_predictions') if os.path.exists(PREDICTIONS_CSV) else pd.DataFrame()

# Parse datetimes
for df_, col in [(df_parking,'datetime'),(df_clustered,'datetime'),(df_preds,'datetime')]:
    if col in df_.columns:
        df_[col] = pd.to_datetime(df_[col], errors='coerce')

Loaded parking_violations: (298450, 31)
Loaded parking_violations_with_clusters: (298450, 32)
Loaded hotspot_clusters: (349, 11)
Loaded congestion_risk_scores: (349, 19)
Loaded enforcement_priority_zones: (349, 19)
Loaded ml_predictions: (298450, 10)


## 4. Build Final Summary Table

In [5]:
if not df_risk.empty and not df_hotspots.empty:
    # Merge risk info into hotspots
    risk_cols = [c for c in ['cluster_id','risk_score','risk_category','recommended_enforcement_action'] if c in df_risk.columns]
    summary = df_hotspots.merge(df_risk[risk_cols], on='cluster_id', how='left')

    final_cols = [c for c in [
        'cluster_id',
        'most_common_police_station',
        'most_common_location_or_junction',
        'violation_count',
        'peak_hour',
        'risk_score',
        'risk_category',
        'recommended_enforcement_action'
    ] if c in summary.columns]

    final_summary = summary[final_cols].sort_values('risk_score', ascending=False).reset_index(drop=True)
    final_summary.to_csv(FINAL_CSV, index=False)
    print(f'Final summary saved: {FINAL_CSV}')
    print(f'Shape: {final_summary.shape}')
    display(final_summary.head(20))
else:
    print('WARNING: Risk or hotspot data missing. Skipping final summary table.')
    final_summary = pd.DataFrame()

Final summary saved: ../data/processed/final_summary_table.csv
Shape: (349, 8)


,cluster_id,most_common_police_station,most_common_location_or_junction,violation_count,peak_hour,risk_score,risk_category,recommended_enforcement_action
0,3,Upparpet,"Kamaraj Road, Sri Nagamma Devi Circle, Sivanch...",186628,5.0,67.00,High,Increase patrolling near junction/station; ins...
1,4,K.R. Pura,"Mbt Road, Devasandra Junction, Kr Puram, Benga...",5596,19.0,39.56,Medium,Periodic patrolling + no-parking awareness drive
2,248,Byatarayanapura,"8Th Cross Road, Roshan Nagar, Deepanjali Nagar...",5,21.0,38.50,Medium,Periodic patrolling + no-parking awareness drive
3,25,Banaswadi,"Old Madras Road, Gopalan Signature Mall, Nagav...",956,9.0,38.12,Medium,Monitor during evening commercial peak; consid...
4,13,Mahadevapura,"Outer Ring Road, Venkatappa Colony, Mahadevapu...",5871,4.0,37.97,Medium,Periodic patrolling + no-parking awareness drive
5,40,Chikkajala,"Bengaluru Bellary Road, Sadahalli Gate Junctio...",959,20.0,37.87,Medium,Periodic patrolling + no-parking awareness drive
6,14,Chikkajala,"Unnamed Road, Begur Chikkanahalli, Bengaluru, ...",5145,23.0,37.77,Medium,Periodic patrolling + no-parking awareness drive
7,6,Hal Old Airport,"New Horizon College Road, New Horizon College ...",20204,22.0,37.71,Medium,Periodic patrolling + no-parking awareness drive
8,68,Peenya,"Jalahalli Cross Road, Jalahalli Cross Junction...",552,6.0,37.41,Medium,Periodic patrolling + no-parking awareness drive
9,108,Hennuru,"Thanisandra Main Road, Aswath Nagar Circle, Th...",162,8.0,36.97,Medium,Periodic patrolling + no-parking awareness drive


## 5. Risk Category Distribution Chart

In [6]:
if not df_risk.empty and 'risk_category' in df_risk.columns:
    cat_order  = ['Critical','High','Medium','Low']
    cat_colors = ['#8B0000','#FF4500','#FFA500','#32CD32']
    cat_counts = df_risk['risk_category'].value_counts().reindex(cat_order, fill_value=0)

    fig, ax = plt.subplots(figsize=(8, 5))
    bars = ax.bar(cat_counts.index, cat_counts.values, color=cat_colors)
    for bar, val in zip(bars, cat_counts.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                str(val), ha='center', fontweight='bold', fontsize=12)
    ax.set_title('Risk Category Distribution of Parking Hotspots', fontsize=14, fontweight='bold')
    ax.set_xlabel('Risk Category')
    ax.set_ylabel('Number of Clusters')
    plt.tight_layout()
    path = os.path.join(CHARTS_DIR, 'risk_category_distribution.png')
    plt.savefig(path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f'Risk distribution chart saved: {path}')
else:
    print('Skipping risk distribution chart.')

Risk distribution chart saved: ../data/processed/charts/risk_category_distribution.png


## 6. Key Project Summary

In [7]:
total_parking   = len(df_parking)
total_clusters  = df_hotspots['cluster_id'].nunique() if 'cluster_id' in df_hotspots.columns else 0
high_risk_zones = int((df_risk['risk_category'] == 'High').sum()) if 'risk_category' in df_risk.columns else 0
crit_risk_zones = int((df_risk['risk_category'] == 'Critical').sum()) if 'risk_category' in df_risk.columns else 0
top_ps          = df_parking['police_station'].mode()[0] if 'police_station' in df_parking.columns and df_parking['police_station'].notna().sum()>0 else 'N/A'
top_loc         = df_parking['location_or_junction'].mode()[0] if 'location_or_junction' in df_parking.columns and df_parking['location_or_junction'].notna().sum()>0 else 'N/A'
peak_hour       = int(df_parking['hour'].mode()[0]) if 'hour' in df_parking.columns and df_parking['hour'].notna().sum()>0 else 'N/A'

print('=' * 60)
print('PROJECT SUMMARY')
print('=' * 60)
print(f'  Total parking violations   : {total_parking:,}')
print(f'  Total hotspot clusters     : {total_clusters}')
print(f'  High-risk zones            : {high_risk_zones}')
print(f'  Critical-risk zones        : {crit_risk_zones}')
print(f'  Top police station         : {top_ps}')
print(f'  Top location/junction      : {top_loc}')
print(f'  Peak enforcement hour      : {peak_hour}:00')
print('=' * 60)

PROJECT SUMMARY
  Total parking violations   : 298,450
  Total hotspot clusters     : 349
  High-risk zones            : 1
  Critical-risk zones        : 0
  Top police station         : Upparpet
  Top location/junction      : Unnamed Road, Begur Chikkanahalli, Bengaluru, Karnataka. Pin-562149 (India)
  Peak enforcement hour      : 5:00


## 7. Helper – Get Map Center

In [8]:
def get_map_center(df, lat_col='latitude', lon_col='longitude', fallback=(28.6139, 77.2090)):
    """Return [lat, lon] center from dataframe; fallback to Delhi coords."""
    if lat_col in df.columns and lon_col in df.columns:
        df_ = df.dropna(subset=[lat_col, lon_col])
        if len(df_) > 0:
            return [df_[lat_col].mean(), df_[lon_col].mean()]
    return list(fallback)

map_center = get_map_center(df_parking)
print(f'Map center: {map_center}')

Map center: [np.float64(12.980801878709853), np.float64(77.60051221133006)]


## 8. Map 1 – Parking Violation Heatmap

In [9]:
if not df_parking.empty and 'latitude' in df_parking.columns and 'longitude' in df_parking.columns:
    df_heat = df_parking.dropna(subset=['latitude','longitude'])

    # Sample up to 50k points for performance
    if len(df_heat) > 50000:
        df_heat = df_heat.sample(50000, random_state=42)
        print(f'Sampled 50,000 points for heatmap rendering.')

    heat_data = df_heat[['latitude','longitude']].values.tolist()

    m_heat = folium.Map(location=map_center, zoom_start=12,
                        tiles='CartoDB positron')
    HeatMap(
        heat_data,
        radius=12,
        blur=10,
        max_zoom=15,
        gradient={0.2: 'blue', 0.5: 'yellow', 0.8: 'orange', 1.0: 'red'}
    ).add_to(m_heat)

    folium.LayerControl().add_to(m_heat)

    heatmap_path = os.path.join(MAPS_DIR, 'parking_heatmap.html')
    m_heat.save(heatmap_path)
    print(f'Parking heatmap saved: {heatmap_path}')
else:
    print('Skipping heatmap: parking data or coordinates not available.')

Sampled 50,000 points for heatmap rendering.
Parking heatmap saved: ../data/processed/maps/parking_heatmap.html


## 9. Map 2 – Hotspot Cluster Map

In [10]:
SEVERITY_COLORS = {
    'Critical': '#8B0000',
    'High':     '#FF4500',
    'Medium':   '#FFA500',
    'Low':      '#32CD32',
}

if not df_hotspots.empty and 'center_latitude' in df_hotspots.columns:
    df_hs = df_hotspots.dropna(subset=['center_latitude','center_longitude'])

    m_hs = folium.Map(location=map_center, zoom_start=12, tiles='CartoDB positron')

    max_count = df_hs['violation_count'].max() if 'violation_count' in df_hs.columns else 1

    for _, row in df_hs.iterrows():
        vc        = row.get('violation_count', 1)
        sev       = str(row.get('severity_level', 'Low'))
        color     = SEVERITY_COLORS.get(sev, '#3388ff')
        radius    = max(5, min(30, int(vc / max_count * 30) + 5))

        popup_html = f"""
        <b>Cluster ID:</b> {row.get('cluster_id','N/A')}<br>
        <b>Violations:</b> {vc:,}<br>
        <b>Police Station:</b> {row.get('most_common_police_station','N/A')}<br>
        <b>Location:</b> {row.get('most_common_location_or_junction','N/A')}<br>
        <b>Peak Hour:</b> {row.get('peak_hour','N/A')}:00<br>
        <b>Severity:</b> {sev}
        """

        folium.CircleMarker(
            location=[row['center_latitude'], row['center_longitude']],
            radius=radius,
            color=color,
            fill=True,
            fill_color=color,
            fill_opacity=0.7,
            popup=folium.Popup(popup_html, max_width=300)
        ).add_to(m_hs)

    # Legend
    legend_html = """
    <div style='position:fixed;bottom:30px;left:30px;z-index:1000;background:white;
                padding:10px;border:2px solid grey;border-radius:5px;font-size:13px'>
    <b>Severity Level</b><br>
    <span style='color:#8B0000'>&#9679;</span> Critical<br>
    <span style='color:#FF4500'>&#9679;</span> High<br>
    <span style='color:#FFA500'>&#9679;</span> Medium<br>
    <span style='color:#32CD32'>&#9679;</span> Low
    </div>
    """
    m_hs.get_root().html.add_child(folium.Element(legend_html))

    cluster_map_path = os.path.join(MAPS_DIR, 'hotspot_cluster_map.html')
    m_hs.save(cluster_map_path)
    print(f'Hotspot cluster map saved: {cluster_map_path}')
else:
    print('Skipping cluster map: hotspot data not available.')

Hotspot cluster map saved: ../data/processed/maps/hotspot_cluster_map.html


## 10. Map 3 – Enforcement Priority Map

In [11]:
RISK_COLORS = {
    'Low':      'green',
    'Medium':   'orange',
    'High':     'red',
    'Critical': 'darkpurple',
}

if not df_priority.empty and 'center_latitude' in df_priority.columns:
    df_ep = df_priority.dropna(subset=['center_latitude','center_longitude'])

    m_ep = folium.Map(location=map_center, zoom_start=12, tiles='CartoDB positron')

    for _, row in df_ep.iterrows():
        risk_cat  = str(row.get('risk_category', 'Low'))
        mk_color  = RISK_COLORS.get(risk_cat, 'blue')

        popup_html = f"""
        <b>Cluster ID:</b> {row.get('cluster_id','N/A')}<br>
        <b>Risk Score:</b> {row.get('risk_score', 'N/A')}<br>
        <b>Risk Category:</b> {risk_cat}<br>
        <b>Violations:</b> {row.get('violation_count','N/A')}<br>
        <b>Peak Hour:</b> {row.get('peak_hour','N/A')}:00<br>
        <b>Action:</b> {row.get('recommended_enforcement_action','N/A')}
        """

        folium.Marker(
            location=[row['center_latitude'], row['center_longitude']],
            icon=folium.Icon(color=mk_color, icon='exclamation-sign', prefix='glyphicon'),
            popup=folium.Popup(popup_html, max_width=320)
        ).add_to(m_ep)

    # Legend
    legend_html = """
    <div style='position:fixed;bottom:30px;left:30px;z-index:1000;background:white;
                padding:10px;border:2px solid grey;border-radius:5px;font-size:13px'>
    <b>Risk Category</b><br>
    <span style='color:green'>&#9679;</span> Low<br>
    <span style='color:orange'>&#9679;</span> Medium<br>
    <span style='color:red'>&#9679;</span> High<br>
    <span style='color:purple'>&#9679;</span> Critical
    </div>
    """
    m_ep.get_root().html.add_child(folium.Element(legend_html))

    priority_map_path = os.path.join(MAPS_DIR, 'enforcement_priority_map.html')
    m_ep.save(priority_map_path)
    print(f'Enforcement priority map saved: {priority_map_path}')
else:
    print('Skipping enforcement priority map: data not available.')

Enforcement priority map saved: ../data/processed/maps/enforcement_priority_map.html


## 11. Final File Output Index

In [12]:
csv_outputs = [
    '../data/processed/cleaned_violations.csv',
    '../data/processed/parking_violations.csv',
    '../data/processed/parking_violations_with_clusters.csv',
    '../data/processed/hotspot_clusters.csv',
    '../data/processed/congestion_risk_scores.csv',
    '../data/processed/enforcement_priority_zones.csv',
    '../data/processed/ml_predictions.csv',
    '../data/processed/final_summary_table.csv',
]

map_outputs = [
    '../data/processed/maps/parking_heatmap.html',
    '../data/processed/maps/hotspot_cluster_map.html',
    '../data/processed/maps/enforcement_priority_map.html',
]

print('=== CSV Outputs ===')
for f in csv_outputs:
    exists = os.path.exists(f)
    size   = f'{os.path.getsize(f)/1024:.1f} KB' if exists else '---'
    print(f'  {"OK" if exists else "MISSING":7s}  {size:12s}  {f}')

print('\n=== Map Outputs ===')
for f in map_outputs:
    exists = os.path.exists(f)
    size   = f'{os.path.getsize(f)/1024:.1f} KB' if exists else '---'
    print(f'  {"OK" if exists else "MISSING":7s}  {size:12s}  {f}')

=== CSV Outputs ===
  OK       116682.1 KB   ../data/processed/cleaned_violations.csv
  OK       116682.1 KB   ../data/processed/parking_violations.csv
  OK       117340.9 KB   ../data/processed/parking_violations_with_clusters.csv
  OK       65.4 KB       ../data/processed/hotspot_clusters.csv
  OK       94.0 KB       ../data/processed/congestion_risk_scores.csv
  OK       94.0 KB       ../data/processed/enforcement_priority_zones.csv
  OK       57131.1 KB    ../data/processed/ml_predictions.csv
  OK       61.9 KB       ../data/processed/final_summary_table.csv

=== Map Outputs ===
  OK       1271.2 KB     ../data/processed/maps/parking_heatmap.html
  OK       463.0 KB      ../data/processed/maps/hotspot_cluster_map.html
  OK       479.0 KB      ../data/processed/maps/enforcement_priority_map.html
